## Architecture reference for this lab

**Step 16 — API Gateway Endpoint**

![Step 16 — API Gateway Endpoint](images/step-16-api-gateway.png)

**Step 17 — Frontend (S3 + CloudFront)**

![Step 17 — Frontend (S3 + CloudFront)](images/step-17-frontend-cloudfront.png)



# Lab 6 — API Gateway + Lambda Proxy + Frontend

**What this lab is.** We give CareConnect a front door for the outside world: a public web
**API** (API Gateway), a small **proxy** function that forwards requests to the deployed
Supervisor, and a simple **web chat page** (Streamlit) to talk to it.

**Why we do it.** The deployed Supervisor from lab-05 can only be called with AWS credentials.
Real users (and a web page) need a normal web address to send questions to. The API + proxy
provide that; the Streamlit page provides something a human can actually click and type into.

**Why it's needed here.** Without this, CareConnect is invisible to users. This lab is what
turns it from a backend service into something you can demonstrate in a browser.

**How it helps the project.** It completes the path: browser → API → proxy → Supervisor →
answer, and back again.

**The use case.** A staff member opens the chat page and types "What are the visiting hours?"
and gets an answer — without touching any code.

> **Security note:** for simplicity the API here has no login (`authorizationType=NONE`) and
> no WAF/rate-limiting. That's fine for a private demo with synthetic data, but a real
> deployment must add authentication (e.g. Amazon Cognito) and WAF before going live.

---

## Prerequisite — run these two setup cells first (every lab has them)

Before this lab's own steps, run the **two setup cells** below. Every notebook (lab-00
through lab-08) starts with these same two cells — on purpose, not by mistake.

**Why they repeat in every notebook.** Each notebook runs in its own fresh "kernel"
(a separate Python session) with no memory of the other notebooks. So each one has to set
itself up from scratch. These two cells are that setup.

**Cell 1 — "bootstrap":** finds the project's main folder (the one containing `lab_helpers`)
no matter where the notebook is opened from, so `import lab_helpers...` always works.

**Cell 2 — "preflight":** checks all helper files are present before the lab begins, and stops
with a clear message if anything is missing — instead of failing confusingly later.

**Do I run them?** Yes — run both, in order, at the top of **every** lab. They take a second
and prevent the most common setup problems. After these two, continue with the lab's steps.

In [1]:
# WHAT THIS CELL DOES (plain English):
# - It looks at the current folder, then its parent, then its parent's parent, and so on,
#   until it finds the folder that contains "lab_helpers". That folder is our project root.
# - It then switches into that folder and adds it to Python's search path, so that
#   'import lab_helpers...' works from anywhere.
# - If it never finds "lab_helpers", it stops with a clear message instead of a confusing error later.
# You do not need to edit anything here — just run it first.
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

Repo root: /home/sagemaker-user/careconnect-patient-assistant-k21


In [2]:
# WHAT THIS CELL DOES (plain English):
# - It checks that each helper file we rely on is actually present on disk.
# - If any are missing, it stops now with a clear list of what to upload, rather than
#   failing in the middle of the lab.
# Run this straight after the bootstrap cell above.
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

Preflight OK — all helper files present.


### Step 1 — Create the proxy function

**What:** create a small AWS Lambda that receives a web request, forwards the question to the
deployed Supervisor, tidies the response, and sends it back (with CORS headers so a browser
can call it).

**Why:** API Gateway can't call the AgentCore Runtime directly, and browsers can't hold AWS
credentials. This proxy sits in the middle: it's the safe translator between the public web
and our private Supervisor.

In [3]:
# WHAT THIS CELL DOES (plain English):
# - Defines the proxy Lambda's code as text. In plain terms the proxy:
#     * accepts a web request containing {"prompt": "...question..."}
#     * calls our deployed Supervisor with that question
#     * cleans up the streamed response into a single answer
#     * returns it with CORS headers so a browser is allowed to read it
# - Then it zips the code, creates the proxy's permission role (which can invoke the runtime),
#   and uploads it as a Lambda function.
import boto3, io, zipfile, json
import lab_helpers.utils as u
lambda_client = boto3.client("lambda", region_name=u.REGION)
account = u.get_aws_account_id()
runtime_arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn")

PROXY_SRC = f'''
import json, re, boto3
AGENT_RUNTIME_ARN = "{runtime_arn}"
AWS_REGION = "{u.REGION}"
CORS = {{"Content-Type":"application/json","Access-Control-Allow-Origin":"*",
        "Access-Control-Allow-Headers":"Content-Type","Access-Control-Allow-Methods":"OPTIONS,POST"}}
def _reply(s,o): return {{"statusCode":s,"headers":CORS,"body":json.dumps(o)}}
def _sse(raw):
    out=[]
    for ln in raw.splitlines():
        ln=ln.strip()
        if ln.startswith("data:"):
            try: evt=json.loads(ln[5:].strip())
            except ValueError: continue
            t=evt.get("event",{{}}).get("contentBlockDelta",{{}}).get("delta",{{}}).get("text")
            if t: out.append(t)
    return "".join(out)
def lambda_handler(event, context):
    if event.get("httpMethod")=="OPTIONS": return {{"statusCode":200,"headers":CORS,"body":""}}
    body=event.get("body") or "{{}}"
    body=json.loads(body) if isinstance(body,str) else body
    prompt=(body.get("prompt") or "").strip()
    if not prompt: return _reply(400,{{"error":"Missing prompt"}})
    c=boto3.client("bedrock-agentcore",region_name=AWS_REGION)
    r=c.invoke_agent_runtime(agentRuntimeArn=AGENT_RUNTIME_ARN,
        payload=json.dumps({{"prompt":prompt}}).encode())
    raw=r["response"].read()
    raw=raw.decode() if isinstance(raw,(bytes,bytearray)) else str(raw)
    ans=re.sub(r"<thinking\\b[^>]*>.*?</thinking>","",_sse(raw) or raw,flags=re.DOTALL).strip()
    return _reply(200,{{"answer":ans}})
'''

buf=io.BytesIO()
with zipfile.ZipFile(buf,"w") as z: z.writestr("lambda_function.py", PROXY_SRC)
buf.seek(0)

proxy_role = u._create_role(
    u.name("CareConnectProxyRole"), "lambda.amazonaws.com",
    {"Version":"2012-10-17","Statement":[
        {"Effect":"Allow","Action":["logs:CreateLogGroup","logs:CreateLogStream","logs:PutLogEvents"],"Resource":"*"},
        {"Effect":"Allow","Action":["bedrock-agentcore:InvokeAgentRuntime"],"Resource":"*"}]},
    u.name("CareConnectProxyPolicy"))

try:
    fn=lambda_client.create_function(FunctionName=u.PROXY_LAMBDA, Runtime="python3.12",
        Role=proxy_role, Handler="lambda_function.lambda_handler",
        Code={"ZipFile":buf.read()}, Timeout=300)
    print("Created proxy:", fn["FunctionArn"])
except lambda_client.exceptions.ResourceConflictException:
    fn=lambda_client.get_function(FunctionName=u.PROXY_LAMBDA)["Configuration"]
    print("Reusing proxy:", fn["FunctionArn"])
proxy_arn=fn["FunctionArn"]

Created role CareConnectProxyRole-sdk
Created proxy: arn:aws:lambda:us-east-1:831963379350:function:careconnect-agentcore-proxy-sdk


### Step 2 — Create the public API endpoint

**What:** create an API Gateway REST API with a `POST /careconnect` route, connect it to the
proxy Lambda, deploy it, and save the resulting URL.

**Why:** this is the actual public web address users and the frontend will send questions to.
Saving it to Parameter Store lets the Streamlit app find it automatically.

In [4]:
# WHAT THIS CELL DOES (plain English):
# - Creates the public API and a '/careconnect' route that accepts POST requests.
# - Connects that route to our proxy Lambda and deploys it to a stage called 'prod'.
# - Grants API Gateway permission to call the Lambda.
# - Prints and saves the final public URL (…execute-api…/prod/careconnect).
# - NOTE: 'authorizationType=NONE' means anyone with the URL can call it — add real auth
#   before any real-world use.
apigw = boto3.client("apigateway", region_name=u.REGION)
api = apigw.create_rest_api(name=u.name("careconnect-api"),
                            endpointConfiguration={"types":["REGIONAL"]})
api_id = api["id"]
root = apigw.get_resources(restApiId=api_id)["items"][0]["id"]
res = apigw.create_resource(restApiId=api_id, parentId=root, pathPart="careconnect")["id"]
apigw.put_method(restApiId=api_id, resourceId=res, httpMethod="POST", authorizationType="NONE")
uri = f"arn:aws:apigateway:{u.REGION}:lambda:path/2015-03-31/functions/{proxy_arn}/invocations"
apigw.put_integration(restApiId=api_id, resourceId=res, httpMethod="POST",
                      type="AWS_PROXY", integrationHttpMethod="POST", uri=uri)
apigw.create_deployment(restApiId=api_id, stageName="prod")

account = u.get_aws_account_id()
lambda_client.add_permission(FunctionName=u.PROXY_LAMBDA,
    StatementId="apigw-invoke", Action="lambda:InvokeFunction",
    Principal="apigateway.amazonaws.com",
    SourceArn=f"arn:aws:execute-api:{u.REGION}:{account}:{api_id}/*/POST/careconnect")

invoke_url=f"https://{api_id}.execute-api.{u.REGION}.amazonaws.com/prod/careconnect"
u.put_ssm_parameter(f"{u.SSM_PREFIX}/api_url", invoke_url)
print("API URL:", invoke_url)

API URL: https://t2mqvg85xd.execute-api.us-east-1.amazonaws.com/prod/careconnect


### Step 3 — Test the public endpoint

**What:** send a real HTTP request to the new API and print the answer (with clear error
output if something goes wrong).

**Why:** this confirms the full public path works: web request → proxy → Supervisor → answer.
If it errors, the next cell reads the proxy's logs to show exactly why.

In [6]:
# WHAT THIS CELL DOES (plain English):
# - Sends a test question to the public API URL as a normal web (POST) request.
# - Prints the answer. If the server returns an error, it prints the real error message so you
#   can see what went wrong (instead of a generic failure).
import urllib.request, urllib.error, json

req = urllib.request.Request(
    invoke_url,
    data=json.dumps({"prompt": "How should I prepare for my colonoscopy?"}).encode(),
    headers={"Content-Type": "application/json"}, method="POST")
try:
    print(urllib.request.urlopen(req, timeout=300).read().decode())
except urllib.error.HTTPError as e:
    print("HTTP", e.code)
    print(e.read().decode())   # <-- the real error message from the Lambda

{"answer": "\"Approved Riverside Health information:\\nPassage 1\\n# Upper Endoscopy (Gastroscopy) Preparation (Riverside Health \u2014 Approved Leaflet, v2026-07) Follow any specific instructions from your care team, which take priority over this general leaflet. ## Fasting Do not eat solid food for at least 6 hours before the procedure. You may usually have small sips of water up to 2 hours before, unless your care team tells you otherwise. ## The day of the procedure Wear comfortable clothing. Leave jewellery and valuables at home. Bring your photo ID and insurance card, and arrive 30 minutes early. ## Sedation and going home If you receive sedation, arrange for a responsible adult to take you home. Do not drive, operate machinery, or sign legal documents for the rest of the day. ## Dentures, glasses, contact lenses You may be asked to remove dentures, glasses, or contact lenses before the procedure. Bring a case for them. ## Medication and clinical questions Do not start, stop, or 

In [7]:
# WHAT THIS CELL DOES (plain English):
# - Reads the most recent logs from the proxy Lambda in CloudWatch.
# - This is your debugging tool: if the API ever returns an error, the exact reason shows here.
import boto3, lab_helpers.utils as u
logs = boto3.client("logs", region_name=u.REGION)
group = f"/aws/lambda/{u.PROXY_LAMBDA}"

streams = logs.describe_log_streams(
    logGroupName=group, orderBy="LastEventTime", descending=True, limit=1)["logStreams"]
events = logs.get_log_events(
    logGroupName=group, logStreamName=streams[0]["logStreamName"], limit=50)["events"]
for e in events:
    print(e["message"].rstrip())

INIT_START Runtime Version: python:3.12.mainlinev2.v31	Runtime Version ARN: arn:aws:lambda:us-east-1::runtime:c1ab740f3656a72d7917665a940f8634df245489445f5a660de5a634d06c5433
START RequestId: d2580771-ab96-4f51-ac4d-ddb4ac81d0d3 Version: $LATEST
END RequestId: d2580771-ab96-4f51-ac4d-ddb4ac81d0d3
REPORT RequestId: d2580771-ab96-4f51-ac4d-ddb4ac81d0d3	Duration: 3630.25 ms	Billed Duration: 3944 ms	Memory Size: 128 MB	Max Memory Used: 92 MB	Init Duration: 313.11 ms


### Step 4 — The web chat page (Streamlit)

**What is Streamlit?** Streamlit is a Python tool that turns a short script into a simple web
page — here, a chat box for CareConnect. Our ready-made page is in
`lab_helpers/frontend/app.py`. It reads the API URL from Parameter Store and sends whatever
you type to that API.

**Why we use it here.** Streamlit is the *fastest way to visually test* that CareConnect works
end to end from a browser. It's an **internal testing / demo tool** — quick to launch, no web
development required. It runs inside your SageMaker environment and is reached through
SageMaker's proxy URL, so effectively only *you* (inside your logged-in SageMaker session) can
see it. That's perfect for checking behaviour, but it is **not** a way to publish the app to
outside users.

**How to launch it.** Open a Terminal (File → New → Terminal) and run the two commands below.
The first pulls the API URL from Parameter Store into an environment variable; the second
starts the web page on port 8501.

```bash
export CARECONNECT_API_URL=$(aws ssm get-parameter --name /app/careconnect/agentcore/api_url --query Parameter.Value --output text)
streamlit run lab_helpers/frontend/app.py --server.port 8501
```

Then open it in your browser using your SageMaker studio address followed by
`proxy/8501/` — for example:
`https://<your-studio-domain>/jupyterlab/default/proxy/8501/` (keep the trailing slash).

**Internal testing vs. production — important.** Use Streamlit here only to *confirm it works*.
Because it lives inside your SageMaker session, it can't be shared with real users and the
session expires. For a **production-ready** frontend that outside users can reach, you would
host the web app on **Amazon S3 + CloudFront** (a static, public, always-on website) — which
is exactly what you built in the console version of this lab — and put proper authentication
(e.g. Amazon Cognito) and WAF in front of it. In short: **Streamlit = quick internal check;
S3 + CloudFront = the real, shareable frontend.**

## Lab 6 complete ✅

REST endpoint + proxy + Streamlit UI, all `-sdk`. Add WAF/rate-limiting before real use.